# Chapter 10: Model Context Protocol (MCP)

EliteAI Academy

Explore MCP architecture, tool discovery and execution, server/client examples, FastMCP, access controls, and multiple servers.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.

The in-memory MCP classes illustrate protocol concepts. The FastMCP section generates a real server as a separate optional demo.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time, asyncio
from typing import List, Dict, Any, Optional, Callable
from dataclasses import dataclass, field

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL
    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 512) -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, model=_model, max_tokens=max_tokens, system=system,
                                    messages=[{"role": "user", "content": prompt}])
        return r.content[0].text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply: MCP READY"))

## 1. MCP Architecture — Key Concepts

In [ ]:
architecture = """
╔═══════════════════════════════════════════════════════════════════╗
║                    MCP ARCHITECTURE                               ║
╠═════════════════╦═════════════════╦═════════════════════════════╗  ║
║  LLM            ║  MCP CLIENT     ║  MCP SERVER                 ║  ║
║  (Intelligence) ║  (Orchestrator) ║  (External World Gateway)   ║  ║
╠═════════════════╬═════════════════╬═════════════════════════════╣  ║
║  Decides WHAT   ║  Discovers &    ║  Exposes:                   ║  ║
║  to do          ║  routes calls   ║  • Tools  (functions)       ║  ║
║                 ║                 ║  • Resources (static data)  ║  ║
║                 ║                 ║  • Prompts (templates)      ║  ║
╠═════════════════╩═════════════════╩═════════════════════════════╝  ║
║  TRANSPORT LAYERS                                                  ║
║  Local:  JSON-RPC over STDIO (inter-process)                       ║
║  Remote: HTTP / Server-Sent Events (SSE)                           ║
╠════════════════════════════════════════════════════════════════════╣
║  MCP vs TOOL FUNCTION CALLING                                      ║
║  Tool Calling: one-to-one, vendor-specific, pre-defined            ║
║  MCP:          one-to-many, open standard, dynamically discovered  ║
╚════════════════════════════════════════════════════════════════════╝

  1. DISCOVER: Client asks server → 'what can you do?'
  2. REQUEST:  LLM formulates call → send_email(to, subject, body)
  3. EXECUTE:  Server runs function, calls real API
  4. RESPOND:  Result returned to LLM context
"""
print(architecture)

## 2. Build an MCP Server from Scratch (Pure Python)

Before using frameworks, implement the core MCP server concepts manually to understand what happens under the hood.

In [ ]:
import inspect

@dataclass
class MCPTool:
    'Represents an MCP-exposed tool (function).'
    name:        str
    description: str
    fn:          Callable
    parameters:  Dict[str, str]

    def schema(self) -> dict:
        'JSON schema representation exposed to MCP clients.'
        return {
            "name":        self.name,
            "description": self.description,
            "inputSchema": {
                "type":       "object",
                "properties": {k: {"type": v} for k, v in self.parameters.items()},
                "required":   list(self.parameters.keys())
            }
        }

    def execute(self, **kwargs) -> Any:
        return self.fn(**kwargs)

@dataclass
class MCPResource:
    'Static data exposed by the server.'
    uri:         str
    name:        str
    description: str
    content:     str
    mime_type:   str = "text/plain"

class MCPServer:
    'Minimal MCP server implementation.'
    def __init__(self, name: str, version: str = "1.0.0"):
        self.name      = name
        self.version   = version
        self._tools:     Dict[str, MCPTool]     = {}
        self._resources: Dict[str, MCPResource] = {}
        self.call_log:   List[Dict] = []

    def tool(self, description: str = ""):
        'Decorator to register a function as an MCP tool.'
        def decorator(fn: Callable) -> Callable:
            json_types = {str: "string", int: "integer", float: "number", bool: "boolean", dict: "object", list: "array"}
            hints  = {k: json_types.get(v, "string")
                      for k, v in fn.__annotations__.items() if k != "return"}
            desc   = description or (inspect.getdoc(fn) or fn.__name__)
            mcp_tool = MCPTool(name=fn.__name__, description=desc, fn=fn, parameters=hints)
            self._tools[fn.__name__] = mcp_tool
            print(f"  🔧 Tool registered: {fn.__name__}")
            return fn
        return decorator

    def resource(self, uri: str, name: str, description: str):
        'Decorator to expose a string as an MCP resource.'
        def decorator(fn: Callable) -> Callable:
            content = fn()
            self._resources[uri] = MCPResource(uri, name, description, content)
            print(f"  📄 Resource registered: {uri}")
            return fn
        return decorator

    def list_tools(self) -> List[dict]:
        'MCP: tools/list'
        return [t.schema() for t in self._tools.values()]

    def list_resources(self) -> List[dict]:
        'MCP: resources/list'
        return [{"uri": r.uri, "name": r.name, "description": r.description}
                for r in self._resources.values()]

    def read_resource(self, uri: str) -> str:
        'MCP: resources/read'
        if uri not in self._resources:
            raise ValueError(f"Resource not found: {uri}")
        return self._resources[uri].content

    def call_tool(self, name: str, arguments: dict) -> dict:
        'MCP: tools/call'
        if name not in self._tools:
            return {"isError": True, "content": [{"type": "text", "text": f"Tool '{name}' not found"}]}
        try:
            result = self._tools[name].execute(**arguments)
            self.call_log.append({"tool": name, "args": arguments, "success": True})
            return {"isError": False, "content": [{"type": "text", "text": str(result)}]}
        except Exception as e:
            self.call_log.append({"tool": name, "args": arguments, "success": False, "error": str(e)})
            return {"isError": True, "content": [{"type": "text", "text": str(e)}]}

    def info(self) -> dict:
        return {"name": self.name, "version": self.version,
                "tools": len(self._tools), "resources": len(self._resources)}

print("✅ MCPServer class ready")

## 3. Build a Business Tools MCP Server

In [ ]:
import random, math
from datetime import datetime

business_server = MCPServer(name="BusinessToolsServer", version="1.0.0")
print("Building MCP server...")

@business_server.tool("Get simulated current stock price for a ticker symbol.")
def get_stock_price(ticker: str) -> dict:
    'Returns simulated stock price data.'
    prices = {"AAPL": 189.25, "GOOGL": 175.80, "MSFT": 430.50, "TSLA": 245.10, "AMZN": 195.60}
    price  = prices.get(ticker.upper(), round(random.uniform(50, 500), 2))
    return {"ticker": ticker.upper(), "price": price, "currency": "USD",
            "timestamp": datetime.now().isoformat()[:19]}

@business_server.tool("Send an email to a recipient.")
def send_email(to: str, subject: str, body: str) -> dict:
    'Sends an email (simulated).'
    msg_id = f"MSG-{random.randint(10000, 99999)}"
    print(f"  📧 Email sent to {to}: '{subject}'")
    return {"status": "sent", "message_id": msg_id, "to": to}

@business_server.tool("Query a customer record from the CRM database.")
def query_crm(customer_id: str) -> dict:
    'Retrieves customer data from CRM.'
    db = {
        "C001": {"name": "Phunkie Yusuf", "plan": "Enterprise", "spend": 48500, "since": "2021"},
        "C002": {"name": "Alice Chen",    "plan": "Pro",        "spend": 2400,  "since": "2023"},
        "C003": {"name": "Bob Okafor",    "plan": "Starter",    "spend": 300,   "since": "2024"},
    }
    if customer_id not in db:
        raise ValueError(f"Customer {customer_id} not found")
    return db[customer_id]

@business_server.tool("Evaluate a Python math expression safely.")
def calculate(expression: str) -> dict:
    'Evaluates a mathematical expression.'
    allowed = set("0123456789+-*/().** ")
    if not all(c in allowed for c in expression):
        raise ValueError("Invalid characters in expression")
    result = eval(expression, {"__builtins__": {}}, {"math": math, "abs": abs, "round": round})
    return {"expression": expression, "result": result}

@business_server.tool("Create a support ticket in the helpdesk system.")
def create_ticket(title: str, priority: str, description: str) -> dict:
    'Creates a support ticket.'
    ticket_id = f"TKT-{random.randint(10000, 99999)}"
    print(f"  🎫 Ticket created: {ticket_id} [{priority}] {title}")
    return {"ticket_id": ticket_id, "title": title, "priority": priority, "status": "open"}

@business_server.resource(
    uri="company://policies/refund",
    name="Refund Policy",
    description="Company refund and return policy document"
)
def refund_policy():
    return """# Refund Policy
- Full refund within 30 days of purchase
- 50% refund for 31-90 days
- No refund after 90 days (except defective products)
- Enterprise customers: escalate to account manager
"""

@business_server.resource(
    uri="company://products/catalog",
    name="Product Catalog",
    description="Current product offerings and pricing"
)
def product_catalog():
    return """# Product Catalog 2025
| Product    | Price/mo | Features                    |
|------------|----------|-----------------------------|
| Starter    | $29      | 1 user, 5GB storage         |
| Pro        | $99      | 10 users, 100GB storage     |
| Enterprise | $499     | Unlimited users, 1TB, SLA   |
"""

print(f"\n✅ Server ready: {business_server.info()}")

## 4. MCP Client — Discovery and Tool Execution

In [ ]:
class MCPClient:
    'MCP Client that connects to a server, discovers capabilities,'
    def __init__(self, server: MCPServer):
        self.server = server
        self._tools_cache: Optional[List[dict]] = None

    def discover(self) -> dict:
        'MCP Discovery: list all available tools and resources.'
        tools     = self.server.list_tools()
        resources = self.server.list_resources()
        self._tools_cache = tools
        return {"tools": tools, "resources": resources}

    def tool_manifest(self) -> str:
        'Human-readable tool manifest for LLM system prompt injection.'
        if self._tools_cache is None:
            self.discover()
        return "\n".join(
            f"- {t['name']}: {t['description']} | params: {list(t['inputSchema']['properties'].keys())}"
            for t in self._tools_cache
        )

    def call(self, tool_name: str, **kwargs) -> str:
        'Execute a tool on the server.'
        result = self.server.call_tool(tool_name, kwargs)
        if result["isError"]:
            raise RuntimeError(result["content"][0]["text"])
        return result["content"][0]["text"]

    def read(self, uri: str) -> str:
        'Read a resource from the server.'
        return self.server.read_resource(uri)

client = MCPClient(business_server)
capabilities = client.discover()

print("=== MCP DISCOVERY ===")
print(f"\n🔧 Tools ({len(capabilities['tools'])}):\n{client.tool_manifest()}")
print(f"\n📄 Resources ({len(capabilities['resources'])}):\n" +
      "\n".join(f"  {r['uri']}: {r['description']}" for r in capabilities['resources']))

print("\n=== DIRECT TOOL CALLS ===")
print("Stock price:", client.call("get_stock_price", ticker="AAPL"))
print("Calculator:",  client.call("calculate", expression="2847 * 3961"))
print("CRM lookup:",  client.call("query_crm", customer_id="C001"))

print("\n=== RESOURCE READ ===")
print(client.read("company://policies/refund"))

## 5. LLM-Driven MCP Agent

The LLM decides which MCP tool to call based on the user's natural language request.

In [ ]:
class LLMDrivenMCPAgent:
    'Agent that uses an LLM to decide which MCP tools to call,'
    def __init__(self, client: MCPClient):
        self.client = client

    def _decide_tool_call(self, user_request: str) -> Optional[dict]:
        'LLM decides if a tool is needed and which one.'
        raw = call_llm(
            f"""User request: {user_request}

Available MCP tools:
{self.client.tool_manifest()}

If a tool is needed, return ONLY JSON:
{{"tool": "<tool_name>", "arguments": {{"param": "value", ...}}}}

If no tool needed, return: {{"tool": null}}""",
            system="You are an MCP routing agent. Return only valid JSON."
        )
        try:
            return json.loads(re.sub(r"```[\w]*", "", raw).strip())
        except:
            return {"tool": None}

    def handle(self, user_request: str, verbose: bool = True) -> str:
        if verbose:
            print(f"\n{'='*55}")
            print(f"👤 Request: {user_request}")

        decision = self._decide_tool_call(user_request)
        tool_name = decision.get("tool")
        tool_args = decision.get("arguments", {})

        tool_result = ""
        if tool_name:
            if verbose:
                print(f"  🔧 MCP Tool call: {tool_name}({tool_args})")
            try:
                tool_result = self.client.call(tool_name, **tool_args)
                if verbose:
                    print(f"  📥 Tool result: {tool_result[:150]}")
            except RuntimeError as e:
                tool_result = f"Tool error: {e}"
                if verbose:
                    print(f"  ❌ {tool_result}")
        else:
            if verbose:
                print(f"  💭 No tool needed — LLM answers directly")

        context = f"\nTool result: {tool_result}" if tool_result else ""
        response = call_llm(
            f"User asked: {user_request}{context}\n\nFormulate a helpful, natural response.",
            system="You are a helpful business assistant. Use tool results to give precise answers."
        )

        if verbose:
            print(f"\n🤖 Response: {response}")
        return response

mcp_agent = LLMDrivenMCPAgent(client)

print("=== LLM-DRIVEN MCP AGENT ===")
requests = [
    "What's the current price of Microsoft stock?",
    "Look up customer C002 in our CRM.",
    "What's 15% tip on a $87.50 bill?",
    "What is our refund policy for a 45-day old purchase?",
    "Create a high-priority ticket: payment system down for customer C003.",
]

for req in requests:
    mcp_agent.handle(req)

## 6. Multi-Tool Chain via MCP

A single request may require multiple sequential MCP tool calls.

In [ ]:
import ast

def resolve_step_references(value, results):
    """Resolve whole results, dotted fields, and references inside strings."""
    if not isinstance(value, str):
        return value
    pattern = r"\$step_(\d+)((?:\.[A-Za-z_]\w*)*)"

    def lookup(match):
        number = int(match.group(1))
        if number not in results:
            raise ValueError(f"Step {number} has not completed")
        result = results[number]
        if isinstance(result, str) and result.startswith("ERROR:"):
            raise ValueError(f"Step {number} failed; dependent work cannot run")
        fields = match.group(2).lstrip(".").split(".") if match.group(2) else []
        if fields and isinstance(result, str):
            try:
                result = json.loads(result)
            except json.JSONDecodeError:
                result = ast.literal_eval(result)
        for name in fields:
            if not isinstance(result, dict) or name not in result:
                raise ValueError(f"Step {number} has no field {name!r}")
            result = result[name]
        return result

    exact = re.fullmatch(pattern, value)
    if exact:
        return lookup(exact)
    return re.sub(pattern, lambda match: str(lookup(match)), value)

class ChainedMCPAgent:
    'Agent that plans and executes a chain of MCP tool calls'
    def __init__(self, client: MCPClient):
        self.client = client

    def plan(self, request: str) -> List[dict]:
        'LLM creates a step-by-step tool plan.'
        raw = call_llm(
            f"""Create a step-by-step plan to answer this request using the available tools.

Request: {request}

Available tools:
{self.client.tool_manifest()}

Return ONLY a JSON array of steps. Each step:
{{"step": <int>, "tool": "<tool_name>", "arguments": {{"param": "value"}}, "description": "<why>"}}

If an argument depends on an earlier result, use "$step_N" for the whole result
or "$step_N.field" for a field. CRM records have name, plan, spend, and since fields;
calculator results have expression and result fields. Use only actual field names.""",
            system="You are a planning agent. Return only valid JSON."
        )
        try:
            return json.loads(re.sub(r"```[\w]*", "", raw).strip())
        except:
            return []

    def execute_plan(self, plan: List[dict]) -> dict:
        'Execute each step, resolving dependencies between steps.'
        step_results = {}

        for step in plan:
            step_num = step["step"]
            tool     = step["tool"]
            args     = step.get("arguments", {})
            desc     = step.get("description", "")

            print(f"  [Step {step_num}] {tool}({args}) — {desc}")

            try:
                resolved_args = {k: resolve_step_references(v, step_results) for k, v in args.items()}
                result = self.client.call(tool, **resolved_args)
                step_results[step_num] = result
                print(f"    ✅ {result[:100]}")
            except (RuntimeError, ValueError, KeyError, TypeError) as e:
                step_results[step_num] = f"ERROR: {e}"
                print(f"    ❌ {e}")

        return step_results

    def handle(self, request: str) -> str:
        print(f"\n{'='*55}")
        print(f"👤 Complex request: {request}")

        plan = self.plan(request)
        print(f"\n📋 Execution plan ({len(plan)} steps):")
        for s in plan:
            print(f"  Step {s['step']}: {s['tool']} — {s.get('description', '')}")

        print("\n⚡ Executing...")
        results = self.execute_plan(plan)

        results_text = "\n".join(f"Step {k}: {v}" for k, v in results.items())
        response = call_llm(
            f"Request: {request}\nTool results:\n{results_text}\nProvide a comprehensive final answer."
        )
        print(f"\n🤖 Final answer: {response}")
        return response

chained_agent = ChainedMCPAgent(client)
print("=== CHAINED MCP TOOL CALLS ===")
chained_agent.handle(
    "Look up customer C001, check their total spend, calculate 10% loyalty discount, "
    "then send them an email about their discount."
)

## 7. FastMCP Server — Production Pattern

In [ ]:
FASTMCP_SERVER_CODE = '''
# fastmcp_server.py — run with: python fastmcp_server.py
from fastmcp import FastMCP
import random, math
from datetime import datetime

mcp = FastMCP("BusinessServer")

@mcp.tool
def get_stock_price(ticker: str) -> dict:
    """
    Get the current simulated stock price for a ticker.
    Args:
        ticker: Stock ticker symbol (e.g. AAPL, MSFT)
    Returns:
        dict with ticker, price, currency, timestamp
    """
    prices = {"AAPL": 189.25, "GOOGL": 175.80, "MSFT": 430.50}
    price  = prices.get(ticker.upper(), round(random.uniform(50, 500), 2))
    return {"ticker": ticker.upper(), "price": price, "currency": "USD",
            "timestamp": datetime.now().isoformat()[:19]}

@mcp.tool
def calculate(expression: str) -> dict:
    """
    Safely evaluate a mathematical expression.
    Args:
        expression: A math expression like "2847 * 3961"
    Returns:
        dict with expression and result
    """
    allowed = set("0123456789+-*/().** ")
    if not all(c in allowed for c in expression):
        raise ValueError("Invalid characters")
    result = eval(expression, {"__builtins__": {}}, {"math": math})
    return {"expression": expression, "result": result}

@mcp.resource("company://policies/refund")
def refund_policy() -> str:
    """Company refund policy document"""
    return "Full refund within 30 days. 50% refund 31-90 days. No refund after 90 days."

if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8000)
'''

from pathlib import Path
server_path = Path.cwd() / "fastmcp_server.py"
server_path.write_text(FASTMCP_SERVER_CODE, encoding="utf-8")

print(f"FastMCP server written to {server_path}")
print("In a separate terminal with the course environment activated, run:")
print("  python fastmcp_server.py")
print("MCP endpoint: http://127.0.0.1:8000/mcp")
print("For a real connection use fastmcp.Client with this endpoint; the earlier MCPClient is an in-memory teaching class.")
print(FASTMCP_SERVER_CODE)


## 8. Anthropic SDK — Native MCP Integration

The Anthropic SDK supports MCP natively via `client.beta.messages.create`. Tools are discovered from the MCP server and passed directly to the model.

In [ ]:
def mcp_tools_to_anthropic_format(mcp_tools: List[dict]) -> List[dict]:
    "Convert MCP tool schemas to Anthropic's native tool format."
    return [
        {
            "name":         t["name"],
            "description":  t["description"],
            "input_schema": t["inputSchema"]
        }
        for t in mcp_tools
    ]

def anthropic_mcp_agent(user_message: str, mcp_client: MCPClient) -> str:
    'Full Anthropic-native MCP agent loop:'

    tools       = mcp_client.discover()["tools"]
    anth_tools  = mcp_tools_to_anthropic_format(tools)
    messages    = [{"role": "user", "content": user_message}]

    print(f"\n👤 {user_message}")
    print(f"  📋 {len(anth_tools)} tools available from MCP server")

    for _tool_round in range(8):
        response = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model,
            max_tokens=1024,
            tools=anth_tools,
            messages=messages,
            system="You are a helpful business assistant. Use the available MCP tools when needed."
        )

        if response.stop_reason == "end_turn":
            text = next((b.text for b in response.content if hasattr(b, "text")), "")
            print(f"  🤖 {text}")
            return text

        elif response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 MCP call: {block.name}({block.input})")
                    try:
                        result = mcp_client.call(block.name, **block.input)
                        print(f"     ✅ {result[:80]}")
                    except RuntimeError as e:
                        result = f"Error: {e}"
                        print(f"     ❌ {result}")

                    tool_results.append({
                        "type":        "tool_result",
                        "tool_use_id": block.id,
                        "content":     result
                    })

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user",      "content": tool_results})
        else:
            break

    return "Stopped without a final response (maximum 8 tool rounds)."

print("=== ANTHROPIC NATIVE MCP AGENT ===")
anthropic_requests = [
    "What's TSLA stock price and what would 100 shares cost?",
    "Get customer C002 from CRM and tell me their plan and spend.",
    "What is our refund policy and does it apply if a customer bought 60 days ago?",
]

for req in anthropic_requests:
    anthropic_mcp_agent(req, client)
    print()

## 9. MCP Server with Security — Authentication & Tool Filtering

In [ ]:
class SecureMCPServer(MCPServer):
    'MCP server with authentication and role-based access control.'
    def __init__(self, name: str):
        super().__init__(name)

        self._access_control: Dict[str, List[str]] = {
            "read_only":    ["get_stock_price", "query_crm", "calculate"],
            "read_write":   ["get_stock_price", "query_crm", "calculate", "send_email", "create_ticket"],
            "admin":        "*",
        }
        self._api_keys: Dict[str, str] = {
            "key_readonly_123":  "read_only",
            "key_readwrite_456": "read_write",
            "key_admin_789":     "admin",
        }

    def authenticate(self, api_key: str) -> Optional[str]:
        return self._api_keys.get(api_key)

    def list_tools_for_role(self, api_key: str) -> List[dict]:
        role = self.authenticate(api_key)
        if not role:
            raise PermissionError("Invalid API key")
        allowed = self._access_control[role]
        if allowed == "*":
            return self.list_tools()
        return [t for t in self.list_tools() if t["name"] in allowed]

    def secure_call(self, api_key: str, tool_name: str, **kwargs) -> dict:
        role = self.authenticate(api_key)
        if not role:
            return {"isError": True, "content": [{"type": "text", "text": "Authentication failed"}]}

        allowed = self._access_control[role]
        if allowed != "*" and tool_name not in allowed:
            return {"isError": True, "content": [{"type": "text",
                    "text": f"Forbidden: role '{role}' cannot call '{tool_name}'"}]}

        return self.call_tool(tool_name, kwargs)

secure_server = SecureMCPServer("SecureBusinessServer")

@secure_server.tool("Get stock price")
def get_stock_price(ticker: str) -> str:
    prices = {"AAPL": 189.25, "MSFT": 430.50}
    return str(prices.get(ticker.upper(), "Not found"))

@secure_server.tool("Send email")
def send_email(to: str, subject: str, body: str) -> str:
    return f"Email sent to {to}: {subject}"

@secure_server.tool("Delete customer record")
def delete_customer(customer_id: str) -> str:
    return f"Customer {customer_id} deleted"

print("=== SECURE MCP SERVER — RBAC ===")
test_keys = [
    ("key_readonly_123",  "read_only",   "get_stock_price",  {"ticker": "AAPL"}),
    ("key_readonly_123",  "read_only",   "send_email",       {"to": "x@y.com", "subject": "hi", "body": "test"}),
    ("key_readwrite_456", "read_write",  "send_email",       {"to": "x@y.com", "subject": "hi", "body": "test"}),
    ("key_admin_789",     "admin",       "delete_customer",  {"customer_id": "C001"}),
    ("bad_key_000",       "INVALID",     "get_stock_price",  {"ticker": "AAPL"}),
]

for api_key, role, tool, args in test_keys:
    result = secure_server.secure_call(api_key, tool, **args)
    icon   = "✅" if not result["isError"] else "❌"
    text   = result["content"][0]["text"]
    print(f"  {icon} [{role}] {tool}: {text[:80]}")

print("\n📋 Tools visible per API key:")
for key, role in [("key_readonly_123", "read_only"), ("key_readwrite_456", "read_write"), ("key_admin_789", "admin")]:
    tools = secure_server.list_tools_for_role(key)
    print(f"  [{role}]: {[t['name'] for t in tools]}")

## 10. Multi-Server MCP Registry

In production, agents connect to multiple specialised MCP servers and discover tools across all of them.

In [ ]:
class MCPRegistry:
    'Registry of multiple MCP servers.'
    def __init__(self):
        self.servers: Dict[str, MCPServer] = {}
        self._tool_index: Dict[str, str] = {}

    def register(self, server: MCPServer):
        self.servers[server.name] = server
        for tool in server.list_tools():
            self._tool_index[tool["name"]] = server.name
        print(f"  🔌 Registered: {server.name} ({len(server._tools)} tools)")

    def list_all_tools(self) -> List[dict]:
        all_tools = []
        for server in self.servers.values():
            for tool in server.list_tools():
                tool["server"] = server.name
                all_tools.append(tool)
        return all_tools

    def call(self, tool_name: str, **kwargs) -> str:
        server_name = self._tool_index.get(tool_name)
        if not server_name:
            raise ValueError(f"Tool '{tool_name}' not found in any registered server")
        result = self.servers[server_name].call_tool(tool_name, kwargs)
        if result["isError"]:
            raise RuntimeError(result["content"][0]["text"])
        return result["content"][0]["text"]

finance_server = MCPServer("FinanceServer")

@finance_server.tool("Get stock price")
def get_stock_price_fin(ticker: str) -> str:
    return f"${ {'AAPL': 189.25, 'MSFT': 430.50}.get(ticker.upper(), 100.0)}"

@finance_server.tool("Calculate compound interest")
def compound_interest(principal: float, rate: float, years: int) -> str:
    result = principal * (1 + rate/100) ** years
    return f"${result:.2f} after {years} years"

hr_server = MCPServer("HRServer")

@hr_server.tool("Look up employee record")
def lookup_employee(employee_id: str) -> str:
    db = {"E001": "Alice Chen | Senior Engineer | Remote",
          "E002": "Bob Okafor | Product Manager | Lagos"}
    return db.get(employee_id, f"Employee {employee_id} not found")

@hr_server.tool("Calculate holiday entitlement")
def holiday_entitlement(years_worked: int) -> str:
    days = 20 + min(years_worked, 5)
    return f"{days} days holiday entitlement"

ops_server = MCPServer("OpsServer")

@ops_server.tool("Check system health")
def check_system_health(service: str) -> str:
    statuses = {"api": "healthy", "database": "healthy", "cache": "degraded"}
    return f"{service}: {statuses.get(service.lower(), 'unknown')}"

@ops_server.tool("Get deployment status")
def deployment_status(environment: str) -> str:
    return f"{environment}: v2.4.1 deployed 2h ago — 99.9% uptime"

registry = MCPRegistry()
print("=== MULTI-SERVER MCP REGISTRY ===")
registry.register(finance_server)
registry.register(hr_server)
registry.register(ops_server)

all_tools = registry.list_all_tools()
print(f"\n📋 {len(all_tools)} tools across {len(registry.servers)} servers:")
for tool in all_tools:
    print(f"  [{tool['server']}] {tool['name']}: {tool['description']}")

print("\n⚡ Cross-server tool calls:")
print("  Finance:", registry.call("get_stock_price_fin", ticker="AAPL"))
print("  HR:",      registry.call("lookup_employee", employee_id="E001"))
print("  Ops:",     registry.call("check_system_health", service="database"))
print("  Finance:", registry.call("compound_interest", principal=10000, rate=8, years=5))

## 11. Quick Reference

In [ ]:
reference = """
╔═════════════════════════╦════════════════════════╦══════════════════════════╗
║ Component               ║ Purpose                ║ Key Consideration        ║
╠═════════════════════════╬════════════════════════╬══════════════════════════╣
║ Tool                    ║ Executable function    ║ Return agent-friendly    ║
║                         ║                        ║ data (not raw bytes/PDF) ║
╠═════════════════════════╬════════════════════════╬══════════════════════════╣
║ Resource                ║ Static data            ║ Use Markdown not PDF     ║
╠═════════════════════════╬════════════════════════╬══════════════════════════╣
║ Prompt                  ║ Interaction template   ║ Guides LLM usage         ║
╠═════════════════════════╬════════════════════════╬══════════════════════════╣
║ STDIO transport         ║ Local/same-machine     ║ Fast, secure, no network ║
╠═════════════════════════╬════════════════════════╬══════════════════════════╣
║ HTTP/SSE transport      ║ Remote/shared server   ║ Scalable, add auth       ║
╠═════════════════════════╬════════════════════════╬══════════════════════════╣
║ FastMCP                 ║ Quick server creation  ║ @mcp.tool decorator,     ║
║                         ║                        ║ auto schema from hints   ║
╠═════════════════════════╬════════════════════════╬══════════════════════════╣
║ MCPRegistry             ║ Multi-server routing   ║ Tool discovery across    ║
║                         ║                        ║ all registered servers   ║
╚═════════════════════════╩════════════════════════╩══════════════════════════╝

USE MCP WHEN:
  ✅ Multiple LLMs need same tools (interoperability)
  ✅ Tools evolve independently of agents (discoverability)
  ✅ Reusing existing internal APIs across many agents
  ✅ Enterprise scale — many teams, many services

USE DIRECT TOOL CALLING WHEN:
  ✅ Simple app, fixed tool set, single LLM provider
  ✅ No need for tool discovery or server separation

⚠️  API DESIGN MATTERS: MCP wraps your API — if the API is bad,
   MCP won't fix it. Add filtering, sorting, pagination at the API level.
"""
print(reference)

## 12. Exercises

- **Exercise 1 — Weather MCP Server:** Build an `MCPServer` with a `get_weather(city: str)` tool that returns simulated weather data, and a `weather://forecasts/cities` resource listing supported cities. Connect it to an `LLMDrivenMCPAgent` and test with "What's the weather in Lagos?"

- **Exercise 2 — Tool Versioning:** Extend `MCPServer` to support versioned tools: `get_stock_price_v1` and `get_stock_price_v2`. Add a `version` field to tool schemas and let the LLM prefer `v2` when available.

- **Exercise 3 — Async MCP:** Rewrite `ChainedMCPAgent.execute_plan` to identify independent steps (no `$step_N` dependencies) and execute them with `asyncio.gather`.

- **Exercise 4 — Error Contract:** Add a standardised error response format to `MCPServer.call_tool`: `{"isError": true, "errorCode": "TOOL_NOT_FOUND"|"VALIDATION_ERROR"|"EXECUTION_FAILED", "message": "..."}`. Update `LLMDrivenMCPAgent` to handle each error code differently.

- **Exercise 5 — FastMCP Client:** Start the `fastmcp_server.py` from Section 7 in a subprocess. Build a client that connects via HTTP, discovers tools, and calls `calculate` through the live server.

**Next:** Chapter 11 — Goal Setting and Monitoring